# Fase 6 — Ambiente Gymnasium

Ambiente RL su barre M15. Le feature vengono costruite direttamente dal prodotto M15 della Fase 2, evitando di usare accidentalmente il dataset M1 della Fase 3 precedente. L'azione osservata a t viene eseguita a t+1.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml
from gymnasium.utils.env_checker import check_env

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from gold_rl.data.features import MARKET_FEATURE_COLUMNS, create_market_features
from gold_rl.execution import ExecutionCostsConfig
from gold_rl.trading_env import TradingEnv

config = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
environment = config['environment']

## 1. M15 + feature causali

Il dataset M15 è il prodotto della Fase 2. Le feature sono ricostruite con la funzione strict della Fase 3; la normalizzazione deve essere applicata con statistiche fit solo sul train quando si prepara il dataset definitivo per l'addestramento.

In [ ]:
m15_path = PROJECT_ROOT / 'data/processed/xauusd_m15.parquet'
if not m15_path.exists():
    raise FileNotFoundError('Manca data/processed/xauusd_m15.parquet: eseguire prima la Fase 2.')

ohlcv = pd.read_parquet(m15_path)
if not isinstance(ohlcv.index, pd.DatetimeIndex):
    raise TypeError('Il dataset M15 deve avere un DatetimeIndex.')

features_all = create_market_features(ohlcv, drop_warmup=True, strict_windows=True)
features_all = features_all.loc[:, list(MARKET_FEATURE_COLUMNS)]
prices = ohlcv.loc[features_all.index, 'Close'].astype('float64')
assert len(features_all) == len(prices) and len(features_all) >= 2
assert np.isfinite(features_all.to_numpy()).all()
assert np.isfinite(prices.to_numpy()).all() and (prices > 0).all()

print('M15:', len(features_all), features_all.index.min(), '->', features_all.index.max())

## 2. Environment e execution engine

L'ambiente riusa execute_order e update_portfolio della Fase 4. Action mapping: 0 short, 1 flat, 2 long.

In [ ]:
costs = ExecutionCostsConfig(
    spread_rate=float(environment['spread_rate']),
    commission_rate=float(environment['commission_rate']),
    slippage_rate=float(environment['slippage_rate']),
    turnover_penalty=float(environment['turnover_penalty']),
)

train_env = TradingEnv(features=features_all, mid_prices=prices, execution_config=costs, initial_balance=float(environment['initial_balance']), mode='train', max_episode_steps=int(environment['max_episode_steps']), turnover_reward_weight=0.0)
validation_env = TradingEnv(features=features_all, mid_prices=prices, execution_config=costs, initial_balance=float(environment['initial_balance']), mode='validation', max_episode_steps=None, random_start=False)
test_env = TradingEnv(features=features_all, mid_prices=prices, execution_config=costs, initial_balance=float(environment['initial_balance']), mode='test', max_episode_steps=None, random_start=False)

print(train_env.action_space)
print(train_env.observation_space)

## 3. Gymnasium check_env

check_env verifica API, spazi, reset, seed e step.

In [ ]:
check_env(train_env, skip_render_check=True)
print('check_env: OK')

## 4. Random agent — episodio completo

L'episodio random deve terminare senza accessi oltre indice e senza valori non finiti.

In [ ]:
obs, info = train_env.reset(seed=int(config['project']['seed']))
steps = 0
while True:
    assert np.isfinite(obs).all()
    obs, reward, terminated, truncated, info = train_env.step(train_env.action_space.sample())
    assert np.isfinite(obs).all() and np.isfinite(reward) and np.isfinite(info['equity'])
    steps += 1
    if terminated or truncated: break
print('Random episode steps:', steps)
print('Final equity:', info['equity'])

## 5. Validation e test cronologici

I due environment partono sempre dall'inizio del segmento e avanzano in ordine temporale.

In [ ]:
for name, env in [('validation', validation_env), ('test', test_env)]:
    _, info = env.reset(seed=123)
    first = info['timestamp']
    _, _, terminated, truncated, info = env.step(1)
    assert info['timestamp'] > first
    assert not truncated
    print(name, first, '->', info['timestamp'])

print('Fase 6 completata.')